# EEG_23 — PLV: Phase Locking Value per C0/C1

**Domanda**: Il meccanismo IS di C0/C1 è visibile nella sincronizzazione di fase tra elettrodi (PLV),
dove la PSD non riesce?

**Tre analisi:**
- **A** — C0 vs C1 (tutti i trial): i fenotipi differiscono in PLV? (cross-validazione indipendente)
- **B** — Dentro C0: corretti vs sbagliati (tutti soggetti C0)
- **C** — Dentro C1: corretti vs sbagliati (tutti soggetti C1) — ipotesi F3↔PO8

**Bande**: alpha (8-13 Hz), beta (13-30 Hz), gamma (30-50 Hz)
Delta/theta escluse: trial da 1.5s troppo brevi per stima affidabile della fase.

## §1 — Setup

In [ ]:
import json, logging, re
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.signal import butter, filtfilt, hilbert
from scipy.stats import mannwhitneyu
from sklearn.metrics import balanced_accuracy_score
from tqdm.auto import tqdm
import mne

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)-8s %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('eeg23')

project_root = Path('/home/daniele_u/miralis-hypergraph-imagined-speech')
FIG_DIR = project_root / 'figures'; FIG_DIR.mkdir(exist_ok=True)
CKPT_13B = project_root / 'models' / 'eeg13b_200e'

N_CHANNELS = 61; N_SAMPLES = 384; N_CLASSES = 4; FS = 256
K_WINDOWS = 8; N_EDGES = 16; D_MODEL = 64; HIDDEN = 128; N_LAYERS = 2; DROPOUT = 0.5
T_WIN = N_SAMPLES // K_WINDOWS; CLUSTER_SCHEME = 'concr4'; DATA_METRIC = 'abs_pcc'
CLUSTER_NAMES = {0: 'C0 Fronto-motor', 1: 'C1 Fronto-occipital'}
PLV_BANDS = {'alpha': (8, 13), 'beta': (13, 30), 'gamma': (30, 50)}
device = torch.device('cpu')

with open(project_root / 'configs' / 'label_schemes' / 'label2idx.json') as f:
    WORD2LABEL = json.load(f)
with open(project_root / 'configs' / 'label_schemes' / f'labelid2cluster_{CLUSTER_SCHEME}.json') as f:
    _raw = json.load(f); label2cluster = {int(k): int(v) for k, v in _raw.items()}
_cd = json.loads((project_root / 'configs' / 'eeg16b_cluster_labels.json').read_text())
SUBJ_CLUSTER = {s: l for s, l in zip(_cd['subj_ids'], _cd['labels'])}

_PAT = re.compile(r'^P(\d+)_S(\d+)$')
subj_sess = defaultdict(lambda: defaultdict(list))
_root = project_root / 'data' / f'hypergraphs_pruned_{DATA_METRIC}'
for p in sorted(_root.rglob('trial_*.pt')):
    m = _PAT.match(p.parent.name)
    if m: subj_sess[int(m.group(1))][int(m.group(2))].append(p)

C0_SUBJ = sorted([s for s, c in SUBJ_CLUSTER.items() if c == 0])
C1_SUBJ = sorted([s for s, c in SUBJ_CLUSTER.items() if c == 1])
ALL_SUBJ = C0_SUBJ + C1_SUBJ
log.info(f'C0={len(C0_SUBJ)} sogg  C1={len(C1_SUBJ)} sogg  totale={len(ALL_SUBJ)}')

CHAN_NAMES_MNE = ['A1','AF7','AF3','Fp1','Fp2','AF4','AF8','A2',
    'F7','F5','F3','F1','F2','F4','F6','F8',
    'FT7','FC5','FC3','FC1','FC2','FC4','FC6','FT8',
    'T7','C5','C3','C1','C2','C4','C6','T8',
    'TP7','CP5','CP3','CP1','CP2','CP4','CP6','TP8',
    'P7','P5','P3','P1','P2','P4','P6','P8',
    'FPz','PO7','PO3','O1','O2','PO4','PO8','Oz',
    'AFz','Fz','FCz','Cz','CPz']
CHAN_IDX = {n: i for i, n in enumerate(CHAN_NAMES_MNE)}

_info = mne.create_info(ch_names=CHAN_NAMES_MNE, sfreq=FS, ch_types='eeg')
_info.set_montage(mne.channels.make_standard_montage('standard_1020'), on_missing='ignore')
HEAD_SCALE = 1.15
# ── Posizioni 2D elettrodi (azimutale equidistante dal montage) ──
# Legge direttamente dal montage, non da info['chs'] che può essere zero
def _make_pos2d(ch_names):
    _mont = mne.channels.make_standard_montage('standard_1020')
    _cp   = _mont.get_positions()['ch_pos']  # dict name -> (x,y,z) in meters
    locs  = np.array([_cp.get(n, np.zeros(3)) for n in ch_names])
    norms = np.linalg.norm(locs, axis=1, keepdims=True).clip(min=1e-10)
    u     = locs / norms
    phi   = np.arccos(u[:, 2].clip(-1, 1))   # angolo dal vertice
    theta = np.arctan2(u[:, 1], u[:, 0])     # azimut
    rho   = phi / (np.pi / 2)
    pos   = np.column_stack([rho * np.cos(theta), rho * np.sin(theta)])
    pr    = pos.max(0) - pos.min(0)
    pos   = (pos - pos.min(0)) / np.where(pr > 0, pr, 1) - 0.5
    return pos * 0.88

POS = _make_pos2d(CHAN_NAMES_MNE)   # (61, 2) — usato da plot_connectome, nice_connectome, labeled_connectome

def _draw_head(ax):
    th = np.linspace(0, 2*np.pi, 200)
    ax.plot(0.5*np.cos(th), 0.5*np.sin(th), 'k-', lw=1.3, zorder=1)
    ax.plot([-.04, 0, .04], [0.495, 0.52, 0.495], 'k-', lw=1.3, zorder=1)

log.info('Setup completato. POS shape: ' + str(POS.shape))


## §2 — DHSLP (identico a EEG_22)

In [ ]:
class HGNNConv(nn.Module):
    def __init__(self,in_dim,out_dim):
        super().__init__()
        self.weight=nn.Parameter(torch.empty(in_dim,out_dim))
        self.bias=nn.Parameter(torch.zeros(out_dim))
        nn.init.xavier_uniform_(self.weight)
    def forward(self,x,H):
        Dv=H.sum(dim=2,keepdim=True).clamp(min=1e-6)
        De=H.sum(dim=1).clamp(min=1e-6).unsqueeze(2)
        Ht=H.transpose(1,2); x_norm=x/Dv
        return torch.bmm(H,torch.bmm(Ht,x_norm)/De)@self.weight+self.bias

class DHSLP(nn.Module):
    def __init__(self,n_nodes=N_CHANNELS,T_win=T_WIN,K=K_WINDOWS,n_edges=N_EDGES,
                 d_model=D_MODEL,hidden=HIDDEN,n_classes=N_CLASSES,n_layers=N_LAYERS,dropout=DROPOUT):
        super().__init__()
        self.K=K; self.T_win=T_win; self.d_model=d_model
        self.E=nn.Parameter(torch.randn(n_edges,d_model)*0.01)
        self.pos_enc=nn.Parameter(torch.randn(n_nodes,d_model)*0.01)
        self.node_proj=nn.Sequential(nn.Linear(T_win,d_model),nn.LayerNorm(d_model),nn.ELU())
        dims=[d_model]+[hidden]*n_layers
        self.convs=nn.ModuleList([HGNNConv(dims[i],dims[i+1]) for i in range(n_layers)])
        self.bns=nn.ModuleList([nn.BatchNorm1d(hidden) for _ in range(n_layers)])
        self.drop=nn.Dropout(dropout); self.clf=nn.Linear(hidden,n_classes)
    def build_dynamic_H(self,feat):
        return torch.softmax(torch.matmul(feat,self.E.T)/(self.d_model**0.5),dim=-1)
    def forward(self,x):
        B,N,T=x.shape
        wins=x.reshape(B,N,self.K,self.T_win).permute(0,2,1,3).reshape(B*self.K,N,self.T_win)
        feat=self.node_proj(wins)+self.pos_enc.unsqueeze(0)
        H=self.build_dynamic_H(feat); h=feat
        for conv,bn in zip(self.convs,self.bns):
            h=conv(h,H); h=bn(h.reshape(-1,h.shape[-1])).reshape(h.shape); h=F.elu(h); h=self.drop(h)
        return self.clf(h.mean(dim=1).reshape(B,self.K,-1).mean(dim=1))

def load_model(sid):
    p = CKPT_13B / f'P{sid:03d}.pt'
    if not p.exists(): return None
    ck = torch.load(p, map_location=device, weights_only=False)
    m = DHSLP().to(device); m.load_state_dict(ck['state_dict']); m.eval(); return m

log.info('DHSLP pronto.')


## §3 — Inference su TUTTI i soggetti C0+C1

Raccoglie: segnale grezzo, cluster, correct flag, per ogni trial.

In [ ]:
def run_inference_all(subj_ids):
    """Inference su tutti i soggetti. Restituisce lista di record con segnale grezzo incluso."""
    records = []
    for sid in tqdm(subj_ids, desc='Inference'):
        cl = SUBJ_CLUSTER.get(sid)
        if cl is None: continue
        model = load_model(sid)
        if model is None: continue
        with torch.no_grad():
            for sess_id, paths in sorted(subj_sess[sid].items()):
                for p in paths:
                    d = torch.load(p, weights_only=False)
                    x = d['x'].float()  # (61, 384)
                    y_word = int(d['y'].squeeze()) if isinstance(d['y'], torch.Tensor) else int(d['y'])
                    y_true = label2cluster.get(y_word)
                    if y_true is None: continue
                    x_norm = (x - x.mean(1, keepdim=True)) / (x.std(1, keepdim=True) + 1e-6)
                    y_pred = int(F.softmax(model(x_norm.unsqueeze(0)), dim=1).squeeze().cpu().numpy().argmax())
                    records.append({
                        'subj_id': sid, 'cluster': cl, 'sess_id': sess_id,
                        'y_true': y_true, 'y_pred': y_pred,
                        'correct': int(y_true == y_pred),
                        'x': x.numpy()  # (61, 384) — segnale grezzo per PLV
                    })
        del model

    log.info(f'Record totali: {len(records)}')
    c0 = [r for r in records if r['cluster']==0]
    c1 = [r for r in records if r['cluster']==1]
    log.info(f'C0: {len(c0)} trial ({sum(r["correct"] for r in c0)} corretti)')
    log.info(f'C1: {len(c1)} trial ({sum(r["correct"] for r in c1)} corretti)')
    return records

ALL_RECORDS = run_inference_all(ALL_SUBJ)


## §4 — PLV computation

Per ogni trial: bandpass → Hilbert → fase → PLV matrix (61×61) per banda.

In [ ]:
def bandpass_filter(x, lo, hi, fs=FS, order=4):
    """x: (61, T) → filtered (61, T)"""
    nyq = fs / 2
    b, a = butter(order, [lo / nyq, hi / nyq], btype='band')
    return filtfilt(b, a, x, axis=1)

def compute_plv_matrix(x_np, band):
    """
    x_np: (61, 384)
    band: (lo, hi) Hz
    Returns PLV matrix (61, 61) — valori in [0, 1]
    """
    lo, hi = band
    x_filt = bandpass_filter(x_np, lo, hi)          # (61, T)
    analytic = hilbert(x_filt, axis=1)               # (61, T) complex
    phase = np.exp(1j * np.angle(analytic))          # unit complex
    # PLV[i,j] = |mean_t( phase[i,t] * conj(phase[j,t]) )|
    plv = np.abs(phase @ phase.conj().T) / x_np.shape[1]
    np.fill_diagonal(plv, 0.0)                       # no self-coupling
    return plv.astype(np.float32)

def compute_plv_all_bands(x_np):
    return {band: compute_plv_matrix(x_np, lo_hi) for band, lo_hi in PLV_BANDS.items()}

# Test su un trial
_test = ALL_RECORDS[0]
_plv = compute_plv_all_bands(_test['x'])
log.info(f'PLV shape per banda: {_plv["alpha"].shape}  range alpha: [{_plv["alpha"].min():.3f}, {_plv["alpha"].max():.3f}]')


## §5 — Pre-calcolo PLV per tutti i trial

Calcola e archivia la matrice PLV per ogni trial (~50-70K trial — può richiedere 15-20 min).

In [ ]:
log.info('Pre-calcolo PLV per tutti i trial...')
for r in tqdm(ALL_RECORDS, desc='PLV'):
    r['plv'] = compute_plv_all_bands(r['x'])
    del r['x']  # libera memoria (segnale grezzo non più necessario)
log.info('PLV calcolato per tutti i trial.')


## §5b — Checkpoint: salva/carica PLV (evita ricalcolo)

**Runna §5b-save dopo §5, poi usa §5b-load per ricaricare senza ricalcolare.**

In [ ]:
import pickle
CHECKPOINT = project_root / 'data' / 'eeg23_plv_records.pkl'

# ── SAVE (runna dopo §5, una volta sola) ──
with open(CHECKPOINT, 'wb') as f:
    pickle.dump(ALL_RECORDS, f)
log.info(f'Checkpoint salvato: {CHECKPOINT}  ({CHECKPOINT.stat().st_size/1e6:.1f} MB)')

In [ ]:
import pickle
CHECKPOINT = project_root / 'data' / 'eeg23_plv_records.pkl'

# ── LOAD (salta §3/§4/§5 se il checkpoint esiste) ──
# Runna questo invece di §3+§4+§5 per risparmiare 15-20 min
with open(CHECKPOINT, 'rb') as f:
    ALL_RECORDS = pickle.load(f)
c0_all = [r for r in ALL_RECORDS if r['cluster'] == 0]
c1_all = [r for r in ALL_RECORDS if r['cluster'] == 1]
c0_ok  = [r for r in ALL_RECORDS if r['cluster']==0 and r['correct']==1]
c0_no  = [r for r in ALL_RECORDS if r['cluster']==0 and r['correct']==0]
c1_ok  = [r for r in ALL_RECORDS if r['cluster']==1 and r['correct']==1]
c1_no  = [r for r in ALL_RECORDS if r['cluster']==1 and r['correct']==0]
log.info(f'Checkpoint caricato: {len(ALL_RECORDS)} record')
log.info(f'C0={len(c0_all)} trial  C1={len(c1_all)} trial')
log.info(f'C0 ok={len(c0_ok)} no={len(c0_no)}  C1 ok={len(c1_ok)} no={len(c1_no)}')
# Ricava anche F3/PO8 idx (definiti in §10 ma utili subito)
F3_IDX  = CHAN_IDX['F3']
PO8_IDX = CHAN_IDX['PO8']

## §6 — Funzioni di analisi e visualizzazione

In [ ]:
def plv_diff_analysis(records_a, records_b, label_a='A', label_b='B'):
    """
    Confronta PLV di due gruppi di trial per ogni banda e coppia di elettrodi.
    Ritorna: per ogni banda → array (61,61) di Cohen's d e p-values.
    """
    results = {}
    for band in PLV_BANDS:
        plv_a = np.stack([r['plv'][band] for r in records_a])  # (N_a, 61, 61)
        plv_b = np.stack([r['plv'][band] for r in records_b])  # (N_b, 61, 61)
        n_ch = plv_a.shape[1]
        cohd = np.zeros((n_ch, n_ch), dtype=np.float32)
        pval = np.ones((n_ch, n_ch), dtype=np.float32)
        # Solo upper triangle (matrice simmetrica)
        for i in range(n_ch):
            for j in range(i+1, n_ch):
                a_ij = plv_a[:, i, j]
                b_ij = plv_b[:, i, j]
                _, p = mannwhitneyu(a_ij, b_ij, alternative='two-sided')
                ps = np.sqrt((a_ij.std()**2 + b_ij.std()**2) / 2 + 1e-12)
                d = (a_ij.mean() - b_ij.mean()) / ps
                cohd[i, j] = cohd[j, i] = d
                pval[i, j] = pval[j, i] = p
        np.fill_diagonal(pval, 1.0); np.fill_diagonal(cohd, 0.0)
        nsig = int((pval < 0.05).sum() // 2)
        log.info(f'  {band}: {nsig}/1830 coppie sig ({label_a} vs {label_b})')
        results[band] = {'cohd': cohd, 'pval': pval, 'nsig': nsig}
    return results

def per_electrode_summary(cohd_mat, pval_mat, method='mean_abs'):
    """
    Collassa la matrice (61,61) in un vettore per elettrodo (61,).
    method='mean_abs': media del |d| su tutte le coppie significative
    """
    sig = pval_mat < 0.05
    out = np.zeros(N_CHANNELS)
    for i in range(N_CHANNELS):
        pairs = sig[i, :] & (np.arange(N_CHANNELS) != i)
        if pairs.sum() > 0:
            out[i] = np.abs(cohd_mat[i, pairs]).mean()
    return out

def plot_plv_analysis(results, title, fname, vlim=0.3):
    """3 plot per riga (alpha/beta/gamma): heatmap + topomap per-elettrodo."""
    bands = list(PLV_BANDS.keys())
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    fig.patch.set_facecolor('white')
    fig.suptitle(title, fontsize=13, color='black')

    for col, band in enumerate(bands):
        cohd = results[band]['cohd']
        pval = results[band]['pval']
        nsig = results[band]['nsig']

        # Row 0: heatmap 61×61
        ax = axes[0, col]; ax.set_facecolor('white')
        im = ax.imshow(cohd, cmap='RdBu_r', vmin=-vlim, vmax=vlim, aspect='auto')
        ax.set_title(f"{band.capitalize()} — {nsig}/1830 coppie sig\nCohen's d (A−B)", fontsize=9, color='black')
        ax.set_xlabel('Elettrodo', color='black', fontsize=8)
        ax.set_ylabel('Elettrodo', color='black', fontsize=8)
        ax.tick_params(colors='black', labelsize=6)
        plt.colorbar(im, ax=ax, fraction=0.046)

        # Row 1: topomap per-elettrodo
        ax = axes[1, col]; ax.set_facecolor('white')
        elec_vals = per_electrode_summary(cohd, pval)
        mask = elec_vals > 0
        im2, _ = mne.viz.plot_topomap(
            elec_vals, _info, axes=ax, cmap='Reds', vlim=(0, vlim),
            mask=mask, mask_params=dict(marker='o', markerfacecolor='k',
                                         markeredgecolor='k', markersize=5, linewidth=0),
            show=False, sphere=HEAD_SCALE)
        ax.set_title(f'{band.capitalize()} — |d| medio per elettrodo\n(punti neri = ha coppie sig)', fontsize=8, color='black')
        plt.colorbar(im2, ax=ax, fraction=0.046, label='mean |d|')

    plt.tight_layout()
    plt.savefig(FIG_DIR / fname, dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close()
    log.info(f'Salvato: {fname}')

def plot_connectome(results, band, title, fname, top_n=30, vlim=0.25):
    """Disegna le top_n coppie più significative come connettoma."""
    cohd = results[band]['cohd']
    pval = results[band]['pval']

    # Prendi le top_n coppie per |d| tra quelle significative
    sig_mask = pval < 0.05
    pairs = [(i, j, cohd[i,j], pval[i,j])
             for i in range(N_CHANNELS)
             for j in range(i+1, N_CHANNELS)
             if sig_mask[i,j]]
    pairs.sort(key=lambda x: abs(x[2]), reverse=True)
    top_pairs = pairs[:top_n]

    if not top_pairs:
        log.warning(f'Nessuna coppia significativa per {band}')
        return

    fig, ax = plt.subplots(1, 1, figsize=(7, 7))
    fig.patch.set_facecolor('white')

    # Topomap di base (valori zero, solo per il layout)
    mne.viz.plot_topomap(np.zeros(N_CHANNELS), _info, axes=ax,
                          show=False, sphere=HEAD_SCALE,
                          cmap='Greys', vlim=(0, 1))

    # Posizioni degli elettrodi nel piano topomap
    from mne.viz.topomap import _get_pos_outlines
    pos, _ = _get_pos_outlines(_info, picks=None, sphere=HEAD_SCALE)
    pos_arr = pos  # (61, 2)

    norm = plt.Normalize(vmin=-vlim, vmax=vlim)
    cmap = plt.cm.RdBu_r

    for i, j, d, p in top_pairs:
        x_vals = [pos_arr[i, 0], pos_arr[j, 0]]
        y_vals = [pos_arr[i, 1], pos_arr[j, 1]]
        ax.plot(x_vals, y_vals, color=cmap(norm(d)),
                lw=1.5 + 2*abs(d)/vlim, alpha=0.7, zorder=3)

    # Colorbar
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    plt.colorbar(sm, ax=ax, fraction=0.046, label="Cohen's d")

    ax.set_title(f"{title}\n{band.capitalize()} — top {len(top_pairs)} coppie sig (p<0.05)",
                 fontsize=10, color='black')
    plt.tight_layout()
    plt.savefig(FIG_DIR / fname, dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close()
    log.info(f'Salvato: {fname}')

log.info('Funzioni pronte.')


## §7 — Analisi A: C0 vs C1 (tutti i trial)

I due fenotipi differiscono in PLV? Cross-validazione indipendente da abs_pcc.

In [ ]:
log.info('=== ANALISI A: C0 vs C1 (tutti i trial) ===')
c0_all = [r for r in ALL_RECORDS if r['cluster'] == 0]
c1_all = [r for r in ALL_RECORDS if r['cluster'] == 1]
log.info(f'C0: {len(c0_all)} trial   C1: {len(c1_all)} trial')

results_A = plv_diff_analysis(c0_all, c1_all, 'C0', 'C1')
plot_plv_analysis(results_A,
    title='Analisi A — PLV: C0 vs C1 (tutti i trial, tutti i soggetti)\n'
          'rosso=C0>C1  blu=C0<C1  punti neri=p<0.05',
    fname='eeg23_A_c0_vs_c1.png')

# Connectome per alpha e gamma
for band in ['alpha', 'gamma']:
    plot_connectome(results_A, band,
        title='C0 vs C1',
        fname=f'eeg23_A_connectome_{band}.png')


## §8 — Analisi B: dentro C0, corretti vs sbagliati (tutti sogg C0)

In [ ]:
log.info('=== ANALISI B: C0 corretti vs sbagliati ===')
c0_ok = [r for r in ALL_RECORDS if r['cluster']==0 and r['correct']==1]
c0_no = [r for r in ALL_RECORDS if r['cluster']==0 and r['correct']==0]
log.info(f'C0 corretti: {len(c0_ok)}   C0 sbagliati: {len(c0_no)}')

results_B = plv_diff_analysis(c0_ok, c0_no, 'C0-ok', 'C0-no')
plot_plv_analysis(results_B,
    title='Analisi B — PLV: C0 corretti vs sbagliati (tutti sogg C0)\n'
          'rosso=corretti>sbagliati  blu=corretti<sbagliati',
    fname='eeg23_B_c0_correct_vs_wrong.png')

for band in ['alpha', 'gamma']:
    plot_connectome(results_B, band,
        title='C0 corretti vs sbagliati',
        fname=f'eeg23_B_connectome_{band}.png')


## §9 — Analisi C: dentro C1, corretti vs sbagliati (tutti sogg C1)

**Ipotesi principale**: PLV F3↔PO8 più alto nei trial corretti C1.

In [ ]:
log.info('=== ANALISI C: C1 corretti vs sbagliati ===')
c1_ok = [r for r in ALL_RECORDS if r['cluster']==1 and r['correct']==1]
c1_no = [r for r in ALL_RECORDS if r['cluster']==1 and r['correct']==0]
log.info(f'C1 corretti: {len(c1_ok)}   C1 sbagliati: {len(c1_no)}')

results_C = plv_diff_analysis(c1_ok, c1_no, 'C1-ok', 'C1-no')
plot_plv_analysis(results_C,
    title='Analisi C — PLV: C1 corretti vs sbagliati (tutti sogg C1)\n'
          'rosso=corretti>sbagliati  blu=corretti<sbagliati',
    fname='eeg23_C_c1_correct_vs_wrong.png')

for band in ['alpha', 'gamma']:
    plot_connectome(results_C, band,
        title='C1 corretti vs sbagliati',
        fname=f'eeg23_C_connectome_{band}.png')


## §10 — Spotlight F3↔PO8

Analisi dedicata alla coppia hub di C1 su tutte le bande e tutti i gruppi.

In [ ]:
F3_IDX  = CHAN_IDX['F3']
PO8_IDX = CHAN_IDX['PO8']

log.info(f'F3={F3_IDX}  PO8={PO8_IDX}')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor('white')
fig.suptitle('Spotlight F3↔PO8 — PLV per banda e gruppo', fontsize=12, color='black')

groups = [
    (c0_all,  'C0 (tutti)',     '#4A90E2'),
    (c1_all,  'C1 (tutti)',     '#FF8C42'),
    (c0_ok,   'C0 corretti',   '#52B788'),
    (c0_no,   'C0 sbagliati',  '#E63946'),
    (c1_ok,   'C1 corretti',   '#52B788'),
    (c1_no,   'C1 sbagliati',  '#E63946'),
]

for ax, band in zip(axes, PLV_BANDS):
    ax.set_facecolor('white')
    data_per_group = []
    labels = []
    colors_box = []
    for records, label, color in groups:
        vals = np.array([r['plv'][band][F3_IDX, PO8_IDX] for r in records])
        data_per_group.append(vals)
        labels.append(label)
        colors_box.append(color)

    vp = ax.violinplot(data_per_group, positions=range(len(groups)), showmedians=True)
    for body, color in zip(vp['bodies'], colors_box):
        body.set_facecolor(color); body.set_alpha(0.6)
    vp['cmedians'].set_colors('black')

    # Mann-Whitney C1 ok vs C1 no
    _, p_c1 = mannwhitneyu(data_per_group[4], data_per_group[5], alternative='two-sided')
    # Mann-Whitney C0 ok vs C0 no
    _, p_c0 = mannwhitneyu(data_per_group[2], data_per_group[3], alternative='two-sided')

    ax.set_xticks(range(len(groups)))
    ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=8, color='black')
    ax.set_title(f'{band.capitalize()}\nC0 ok vs no: p={p_c0:.4f}\nC1 ok vs no: p={p_c1:.4f}',
                 fontsize=9, color='black')
    ax.set_ylabel('PLV F3↔PO8', color='black', fontsize=9)
    ax.tick_params(colors='black')
    for sp in ax.spines.values(): sp.set_edgecolor('#CCCCCC')

plt.tight_layout()
plt.savefig(FIG_DIR / 'eeg23_spotlight_F3_PO8.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
plt.close()
log.info('Salvato: eeg23_spotlight_F3_PO8.png')


## §11 — Verdict

In [ ]:
print('=' * 60)
print('EEG_23 — VERDICT')
print('=' * 60)
print()
print('Analisi A — C0 vs C1 (tutti i trial):')
for band, res in results_A.items():
    print(f'  {band:8s}: {res["nsig"]:4d}/1830 coppie sig')

print()
print('Analisi B — C0 corretti vs sbagliati:')
for band, res in results_B.items():
    print(f'  {band:8s}: {res["nsig"]:4d}/1830 coppie sig')

print()
print('Analisi C — C1 corretti vs sbagliati:')
for band, res in results_C.items():
    print(f'  {band:8s}: {res["nsig"]:4d}/1830 coppie sig')

print()
print('F3-PO8 specifico:')
for band in PLV_BANDS:
    c1_ok_vals  = np.array([r['plv'][band][F3_IDX, PO8_IDX] for r in c1_ok])
    c1_no_vals  = np.array([r['plv'][band][F3_IDX, PO8_IDX] for r in c1_no])
    _, p = mannwhitneyu(c1_ok_vals, c1_no_vals, alternative='two-sided')
    ps = np.sqrt((c1_ok_vals.std()**2 + c1_no_vals.std()**2)/2 + 1e-12)
    d  = (c1_ok_vals.mean() - c1_no_vals.mean()) / ps
    print(f'  C1 {band:8s} F3-PO8: d={d:+.4f}  p={p:.4f}  ok_mean={c1_ok_vals.mean():.4f}  no_mean={c1_no_vals.mean():.4f}')
print('=' * 60)


## §12 — Plot finali: connectome + topomap puliti

In [ ]:
import matplotlib.gridspec as gridspec

# POS e _draw_head definiti in §1


def hub_scores(cohd, pval):
    h = np.zeros(N_CHANNELS)
    sig = pval < 0.05
    for i in range(N_CHANNELS):
        idx = sig[i, :]
        if idx.sum() > 0:
            h[i] = np.abs(cohd[i, idx]).mean()
    return h

def nice_toporow(results, suptitle, fname, vlim=0.15):
    bands = list(PLV_BANDS.keys())
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
    fig.patch.set_facecolor('white')
    fig.suptitle(suptitle, fontsize=11, color='black', y=1.01)
    for ax, band in zip(axes, bands):
        ax.set_facecolor('white')
        cohd = results[band]['cohd']
        pval = results[band]['pval']
        nsig = results[band]['nsig']
        elec_vals = hub_scores(cohd, pval)
        mask = elec_vals > 0
        im, _ = mne.viz.plot_topomap(
            elec_vals, _info, axes=ax, cmap='Reds', vlim=(0, vlim),
            mask=mask,
            mask_params=dict(marker='o', markerfacecolor='black',
                             markeredgecolor='black', markersize=5, linewidth=0),
            show=False, sphere=HEAD_SCALE)
        ax.set_title(f'{band.capitalize()} - {nsig}/1830 sig  |  mean |d| per elettrodo',
                     fontsize=9, color='black')
        plt.colorbar(im, ax=ax, fraction=0.046, shrink=0.75).ax.tick_params(labelsize=7)
    plt.tight_layout()
    plt.savefig(FIG_DIR / fname, dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close()
    log.info('Salvato: ' + fname)

def nice_connectome(cohd, pval, title, fname, top_n=50, vlim=0.25, node_vmax=0.15):
    sig_mask = pval < 0.05
    pairs = [(i, j, cohd[i,j]) for i in range(N_CHANNELS)
             for j in range(i+1, N_CHANNELS) if sig_mask[i,j]]
    pairs.sort(key=lambda x: abs(x[2]), reverse=True)
    top_pairs = pairs[:top_n]
    hub = hub_scores(cohd, pval)
    fig, ax = plt.subplots(figsize=(6.5, 7))
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white'); ax.set_aspect('equal'); ax.axis('off')
    _draw_head(ax)
    norm_line = plt.Normalize(vmin=-vlim, vmax=vlim)
    cmap_line = plt.cm.RdBu_r
    norm_hub  = plt.Normalize(vmin=0, vmax=node_vmax)
    cmap_hub  = plt.cm.YlOrRd
    for i, j, d in top_pairs:
        color = cmap_line(norm_line(d))
        lw = 0.8 + 2.8*abs(d)/vlim
        ax.plot([POS[i,0], POS[j,0]], [POS[i,1], POS[j,1]],
                color=color, lw=lw, alpha=0.55, zorder=2, solid_capstyle='round')
    for k in range(N_CHANNELS):
        c  = cmap_hub(norm_hub(hub[k])) if hub[k] > 0 else '#DDDDDD'
        sz = 28 + 200*hub[k]/node_vmax  if hub[k] > 0 else 16
        ec = '#444444' if hub[k] > 0 else '#AAAAAA'
        ax.scatter(POS[k,0], POS[k,1], s=sz, c=[c], edgecolors=ec,
                   linewidths=0.5, zorder=3)
    for name in ['F3', 'PO8', 'F6', 'FT8', 'C3', 'CP2']:
        if name in CHAN_IDX:
            idx = CHAN_IDX[name]
            ax.annotate(name, xy=(POS[idx,0], POS[idx,1]),
                        xytext=(POS[idx,0]+0.035, POS[idx,1]+0.035),
                        fontsize=7, color='#111111', zorder=5,
                        arrowprops=dict(arrowstyle='-', color='#888888', lw=0.5))
    sm = plt.cm.ScalarMappable(cmap=cmap_line, norm=norm_line)
    cb = plt.colorbar(sm, ax=ax, fraction=0.035, pad=0.02, shrink=0.55)
    cb.set_label("Cohen's d  (rosso=A>B, blu=A<B)", fontsize=8)
    cb.ax.tick_params(labelsize=7)
    ax.set_title(title + '  |  ' + str(len(pairs)) + '/1830 sig  top-' + str(len(top_pairs)),
                 fontsize=9, color='black', pad=6)
    ax.set_xlim(-0.62, 0.62); ax.set_ylim(-0.6, 0.6)
    plt.tight_layout()
    plt.savefig(FIG_DIR / fname, dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close()
    log.info('Salvato: ' + fname)

log.info('Funzioni plot avanzate pronte. POS shape: ' + str(POS.shape))


In [ ]:
# Analisi A: C0 vs C1
nice_toporow(results_A, suptitle='A - PLV: C0 vs C1 (tutti i trial)', fname='eeg23_A_toporow.png')
for band in ['alpha', 'beta', 'gamma']:
    nice_connectome(results_A[band]['cohd'], results_A[band]['pval'],
        title='A - C0 vs C1 | ' + band.capitalize(),
        fname='eeg23_A_connectome_' + band + '_v2.png')

# Analisi B: C0 corretti vs sbagliati
nice_toporow(results_B, suptitle='B - PLV C0: corretti vs sbagliati', fname='eeg23_B_toporow.png')
for band in ['alpha', 'beta', 'gamma']:
    nice_connectome(results_B[band]['cohd'], results_B[band]['pval'],
        title='B - C0 corretti vs sbagliati | ' + band.capitalize(),
        fname='eeg23_B_connectome_' + band + '_v2.png')

# Analisi C: C1 corretti vs sbagliati
nice_toporow(results_C, suptitle='C - PLV C1: corretti vs sbagliati', fname='eeg23_C_toporow.png')
for band in ['alpha', 'beta', 'gamma']:
    nice_connectome(results_C[band]['cohd'], results_C[band]['pval'],
        title='C - C1 corretti vs sbagliati | ' + band.capitalize(),
        fname='eeg23_C_connectome_' + band + '_v2.png')
print('Tutti i plot singoli generati.')


In [ ]:
# Figure di sintesi: Analisi C - 2 righe x 3 bande
fig = plt.figure(figsize=(17, 9))
fig.patch.set_facecolor('white')
fig.suptitle('EEG_23 - Analisi C: C1 corretti vs sbagliati (PLV)  |  Riga1: hub score  Riga2: connectome',
             fontsize=11, color='black')
bands = list(PLV_BANDS.keys())
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.3)
cmap_line = plt.cm.RdBu_r
norm_line  = plt.Normalize(vmin=-0.25, vmax=0.25)
cmap_hub   = plt.cm.YlOrRd
norm_hub   = plt.Normalize(vmin=0, vmax=0.15)
for col, band in enumerate(bands):
    cohd = results_C[band]['cohd']
    pval = results_C[band]['pval']
    nsig = results_C[band]['nsig']
    # row 0 - topomap
    ax0 = fig.add_subplot(gs[0, col])
    ax0.set_facecolor('white')
    elec_vals = hub_scores(cohd, pval)
    im, _ = mne.viz.plot_topomap(
        elec_vals, _info, axes=ax0, cmap='Reds', vlim=(0, 0.15),
        mask=elec_vals > 0,
        mask_params=dict(marker='o', markerfacecolor='black',
                         markeredgecolor='black', markersize=5, linewidth=0),
        show=False, sphere=HEAD_SCALE)
    ax0.set_title(band.capitalize() + ' - ' + str(nsig) + '/1830 sig  |  mean |d| per elettrodo',
                  fontsize=9, color='black')
    plt.colorbar(im, ax=ax0, fraction=0.046, shrink=0.72).ax.tick_params(labelsize=7)
    # row 1 - connectome
    ax1 = fig.add_subplot(gs[1, col])
    ax1.set_facecolor('white'); ax1.set_aspect('equal'); ax1.axis('off')
    _draw_head(ax1)
    hub = hub_scores(cohd, pval)
    sig_mask = pval < 0.05
    pairs_c = [(i,j,cohd[i,j]) for i in range(N_CHANNELS)
               for j in range(i+1,N_CHANNELS) if sig_mask[i,j]]
    pairs_c.sort(key=lambda x: abs(x[2]), reverse=True)
    for i, j, d in pairs_c[:50]:
        color = cmap_line(norm_line(d))
        lw = 0.7 + 2.5*abs(d)/0.25
        ax1.plot([POS[i,0],POS[j,0]], [POS[i,1],POS[j,1]],
                 color=color, lw=lw, alpha=0.55, zorder=2, solid_capstyle='round')
    for k in range(N_CHANNELS):
        c  = cmap_hub(norm_hub(hub[k])) if hub[k]>0 else '#DDDDDD'
        sz = 25 + 180*hub[k]/0.15 if hub[k]>0 else 14
        ax1.scatter(POS[k,0], POS[k,1], s=sz, c=[c],
                    edgecolors='#444' if hub[k]>0 else '#BBB',
                    linewidths=0.4, zorder=3)
    for name in ['F3', 'PO8']:
        idx = CHAN_IDX[name]
        ax1.annotate(name, xy=(POS[idx,0], POS[idx,1]),
                     xytext=(POS[idx,0]+0.04, POS[idx,1]+0.04),
                     fontsize=8, fontweight='bold', color='#CC0000',
                     arrowprops=dict(arrowstyle='->', color='#CC0000', lw=0.8))
    ax1.set_title(band.capitalize() + ' - top-50 coppie  |  rosso=ok>no  blu=no>ok',
                  fontsize=9, color='black')
    ax1.set_xlim(-0.62, 0.62); ax1.set_ylim(-0.6, 0.6)
plt.savefig(FIG_DIR / 'eeg23_C1_summary_figure.png',
            dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
plt.close()
log.info('Salvato: eeg23_C1_summary_figure.png')
print('Summary figure salvata.')


## §13 — Coppie nominate: tabella + connectome etichettato

Quali specifiche coppie elettrodo↔elettrodo caratterizzano IS in C0 vs C1?

In [ ]:
# ── Tabella top coppie nominate ──────────────────────────────────
def top_pairs_table(results, label, top_n=20):
    print(f'\n{"="*65}')
    print(f'  {label}')
    print(f'{"="*65}')
    for band in PLV_BANDS:
        cohd = results[band]['cohd']
        pval = results[band]['pval']
        nsig = results[band]['nsig']
        sig_mask = pval < 0.05
        pairs = []
        for i in range(N_CHANNELS):
            for j in range(i+1, N_CHANNELS):
                if sig_mask[i, j]:
                    pairs.append((i, j, cohd[i,j], pval[i,j]))
        pairs.sort(key=lambda x: abs(x[2]), reverse=True)
        print(f'\n  {band.upper()} ({nsig}/1830 sig) — top {min(top_n, len(pairs))}:')
        print(f'  {"Rank":<5} {"Coppia":<18} {"d":>8} {"p":>10}  Direzione')
        print(f'  {"-"*60}')
        for rank, (i, j, d, p) in enumerate(pairs[:top_n], 1):
            ei = CHAN_NAMES_MNE[i]
            ej = CHAN_NAMES_MNE[j]
            pair_str = f'{ei} <-> {ej}'
            direction = 'ok > no' if d > 0 else 'no > ok'
            p_str = f'{p:.2e}' if p < 0.001 else f'{p:.4f}'
            print(f'  {rank:<5} {pair_str:<18} {d:>+8.4f} {p_str:>10}  {direction}')

top_pairs_table(results_B, 'B — C0 corretti vs sbagliati')
top_pairs_table(results_C, 'C — C1 corretti vs sbagliati')


In [ ]:
# ── Connectome con nomi elettrodi sulle coppie top ───────────────
def labeled_connectome(cohd, pval, title, fname,
                        top_n=15, vlim=0.25, node_vmax=0.15):
    sig_mask = pval < 0.05
    pairs = [(i, j, cohd[i,j]) for i in range(N_CHANNELS)
             for j in range(i+1, N_CHANNELS) if sig_mask[i,j]]
    pairs.sort(key=lambda x: abs(x[2]), reverse=True)
    top_pairs = pairs[:top_n]
    if not top_pairs:
        log.warning('Nessuna coppia sig'); return
    hub = hub_scores(cohd, pval)
    fig, ax = plt.subplots(figsize=(8, 8.5))
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white'); ax.set_aspect('equal'); ax.axis('off')
    _draw_head(ax)
    norm_line = plt.Normalize(vmin=-vlim, vmax=vlim)
    cmap_line = plt.cm.RdBu_r
    norm_hub  = plt.Normalize(vmin=0, vmax=node_vmax)
    cmap_hub  = plt.cm.YlOrRd
    # linee
    for rank, (i, j, d) in enumerate(top_pairs):
        color = cmap_line(norm_line(d))
        lw = 1.0 + 3.0*abs(d)/vlim
        ax.plot([POS[i,0], POS[j,0]], [POS[i,1], POS[j,1]],
                color=color, lw=lw, alpha=0.65, zorder=2, solid_capstyle='round')
        # etichetta coppia al midpoint
        mx = (POS[i,0] + POS[j,0]) / 2
        my = (POS[i,1] + POS[j,1]) / 2
        ei = CHAN_NAMES_MNE[i]; ej = CHAN_NAMES_MNE[j]
        arrow_col = '#CC2200' if d > 0 else '#0033AA'
        ax.text(mx, my, f'{ei}-{ej}', fontsize=6, color=arrow_col,
                ha='center', va='center', zorder=6,
                bbox=dict(boxstyle='round,pad=0.15', fc='white', ec=arrow_col,
                          lw=0.6, alpha=0.85))
    # nodi
    for k in range(N_CHANNELS):
        c  = cmap_hub(norm_hub(hub[k])) if hub[k] > 0 else '#DDDDDD'
        sz = 30 + 220*hub[k]/node_vmax  if hub[k] > 0 else 18
        ec = '#333333' if hub[k] > 0 else '#BBBBBB'
        ax.scatter(POS[k,0], POS[k,1], s=sz, c=[c], edgecolors=ec,
                   linewidths=0.5, zorder=4)
    # etichette nodi con coppie sig (i top hub)
    top_hub_idx = np.argsort(hub)[::-1][:12]
    for k in top_hub_idx:
        if hub[k] > 0:
            ax.text(POS[k,0], POS[k,1], CHAN_NAMES_MNE[k],
                    fontsize=6.5, fontweight='bold', color='black',
                    ha='center', va='center', zorder=7)
    sm = plt.cm.ScalarMappable(cmap=cmap_line, norm=norm_line)
    cb = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02, shrink=0.5)
    cb.set_label("Cohen's d  (rosso=ok>no, blu=no>ok)", fontsize=8)
    cb.ax.tick_params(labelsize=7)
    total_sig = len(pairs)
    ax.set_title(f'{title}\n{total_sig}/1830 coppie sig  |  top-{len(top_pairs)} etichettate',
                 fontsize=10, color='black', pad=8)
    ax.set_xlim(-0.65, 0.65); ax.set_ylim(-0.62, 0.62)
    plt.tight_layout()
    plt.savefig(FIG_DIR / fname, dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close()
    log.info('Salvato: ' + fname)

# B: C0 per banda
for band in ['alpha', 'beta', 'gamma']:
    labeled_connectome(results_B[band]['cohd'], results_B[band]['pval'],
        title='B — C0 corretti vs sbagliati | ' + band.capitalize(),
        fname='eeg23_B_labeled_' + band + '.png')

# C: C1 per banda
for band in ['alpha', 'beta', 'gamma']:
    labeled_connectome(results_C[band]['cohd'], results_C[band]['pval'],
        title='C — C1 corretti vs sbagliati | ' + band.capitalize(),
        fname='eeg23_C_labeled_' + band + '.png')

print('Connectome etichettati generati.')


In [ ]:
# ── Figura comparativa B vs C (gamma, il band con piu segnale) ───
fig, axes = plt.subplots(1, 2, figsize=(16, 8.5))
fig.patch.set_facecolor('white')
fig.suptitle('Top-15 coppie PLV significative — Gamma  |  L: C0 corretti vs sbagliati  |  R: C1 corretti vs sbagliati',
             fontsize=11, color='black')

for ax, results, lbl, fname_sfx in [
        (axes[0], results_B, 'B — C0', 'B'),
        (axes[1], results_C, 'C — C1', 'C')]:
    cohd = results['gamma']['cohd']
    pval = results['gamma']['pval']
    nsig = results['gamma']['nsig']
    sig_mask = pval < 0.05
    pairs = [(i,j,cohd[i,j]) for i in range(N_CHANNELS)
             for j in range(i+1,N_CHANNELS) if sig_mask[i,j]]
    pairs.sort(key=lambda x: abs(x[2]), reverse=True)
    top15 = pairs[:15]
    hub   = hub_scores(cohd, pval)
    ax.set_facecolor('white'); ax.set_aspect('equal'); ax.axis('off')
    _draw_head(ax)
    norm_line = plt.Normalize(vmin=-0.25, vmax=0.25)
    cmap_line = plt.cm.RdBu_r
    norm_hub  = plt.Normalize(vmin=0, vmax=0.15)
    cmap_hub  = plt.cm.YlOrRd
    for i, j, d in top15:
        color = cmap_line(norm_line(d))
        lw = 1.0 + 3.0*abs(d)/0.25
        ax.plot([POS[i,0],POS[j,0]], [POS[i,1],POS[j,1]],
                color=color, lw=lw, alpha=0.65, zorder=2, solid_capstyle='round')
        mx = (POS[i,0]+POS[j,0])/2; my = (POS[i,1]+POS[j,1])/2
        ei = CHAN_NAMES_MNE[i]; ej = CHAN_NAMES_MNE[j]
        arrow_col = '#CC2200' if d > 0 else '#0033AA'
        ax.text(mx, my, f'{ei}-{ej}', fontsize=6.5, color=arrow_col,
                ha='center', va='center', zorder=6,
                bbox=dict(boxstyle='round,pad=0.15', fc='white', ec=arrow_col,
                          lw=0.6, alpha=0.88))
    for k in range(N_CHANNELS):
        c  = cmap_hub(norm_hub(hub[k])) if hub[k]>0 else '#DDDDDD'
        sz = 30 + 220*hub[k]/0.15 if hub[k]>0 else 18
        ax.scatter(POS[k,0], POS[k,1], s=sz, c=[c],
                   edgecolors='#333' if hub[k]>0 else '#BBB', linewidths=0.5, zorder=4)
    for k in np.argsort(hub)[::-1][:10]:
        if hub[k] > 0:
            ax.text(POS[k,0], POS[k,1], CHAN_NAMES_MNE[k],
                    fontsize=6.5, fontweight='bold', color='black',
                    ha='center', va='center', zorder=7)
    ax.set_title(lbl + ' — Gamma  |  ' + str(nsig) + '/1830 sig  top-15 etichettate',
                 fontsize=10, color='black')
    ax.set_xlim(-0.65, 0.65); ax.set_ylim(-0.62, 0.62)

plt.tight_layout()
plt.savefig(FIG_DIR / 'eeg23_BC_gamma_labeled_comparison.png',
            dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
plt.close()
log.info('Salvato: eeg23_BC_gamma_labeled_comparison.png')
print('Figura comparativa B vs C generata.')


## §14 — Topomap per-elettrodo: hub PLV per effect size

Collassa la matrice PLV (61×61) in per-elettrodo: per ogni canale, banda dominante e direzione (corretti>sbagliati o viceversa), misurate dal **d di Cohen medio** sulle coppie.

**Selezione per effect size, non per p-value.** Con ~38k trial i p-value sono saturi: tutti gli elettrodi risultano "significativi", quindi la soglia su p non seleziona nulla (era 61/61 in ogni pannello). Si mostrano invece i **top-15 elettrodi per |d| medio**.

2 panel: B (C0 corretti vs sbagliati) e C (C1 corretti vs sbagliati). I pannelli C0-vs-C1 su tutti i trial sono ridondanti tra loro (speculari) e omessi.

In [ ]:
import matplotlib.patches as mpatches

# POS e _draw_head definiti in §1 (azimutale equidistante, standard_1020)
# SELEZIONE PER EFFECT SIZE, non per p-value: con ~38k trial i p sono saturi.
# Hub = elettrodi con |d| medio piu' alto sulle coppie. Freccia = direzione netta.

BAND_COLOR = {'alpha': '#FF8C42', 'beta': '#C8920A', 'gamma': '#52B788'}
TOP_K = 15  # quanti elettrodi-hub mostrare per pannello


def electrode_hub_strength(results):
    """Per ogni elettrodo: banda dominante, |d| medio (magnitudine) e d netto (segno).

    Selezione per |d| medio (cattura coinvolgimento anche con direzioni miste).
    Returns {ch_idx: (dominant_band, mean_abs_d, mean_signed_d)}
    """
    out = {}
    for ch in range(N_CHANNELS):
        mask = np.arange(N_CHANNELS) != ch
        band_mag, band_signed = {}, {}
        for band in PLV_BANDS:
            cohd_ch = results[band]['cohd'][ch, mask]
            band_mag[band] = float(np.abs(cohd_ch).mean())
            band_signed[band] = float(cohd_ch.mean())
        dom = max(band_mag, key=lambda b: band_mag[b])
        out[ch] = (dom, band_mag[dom], band_signed[dom])
    return out


def top_electrodes_plv(results, top_k=TOP_K):
    """Top-k elettrodi per |d| medio della banda dominante."""
    strengths = electrode_hub_strength(results)
    ranked = sorted(strengths.items(), key=lambda kv: kv[1][1], reverse=True)
    return dict(ranked[:top_k])


def plot_topo_panel(ax, top, title, mag_max, top_k=TOP_K):
    ax.set_facecolor('white')
    _draw_head(ax)
    ax.scatter(POS[:, 0], POS[:, 1], s=24, c='#E6E6E6',
               edgecolors='#BBBBBB', lw=0.4, zorder=2)
    bands_present = set()
    for ch, (dom, mag, signed) in top.items():
        p = POS[ch]
        size = 90 + 320 * (mag / mag_max)
        arrow = '↑' if signed > 0 else '↓'
        ax.scatter(p[0], p[1], s=size, c=BAND_COLOR[dom],
                   edgecolors='#222222', lw=1.3, zorder=5)
        ax.annotate(f'{CHAN_NAMES_MNE[ch]}{arrow}  |d|={mag:.2f}', (p[0], p[1]),
                    fontsize=7, fontweight='bold', color='#111111',
                    ha='center', va='bottom',
                    xytext=(0, 7), textcoords='offset points', zorder=10)
        bands_present.add(dom)
    ax.set_title(f'{title}\ntop-{top_k} elettrodi per |d| medio',
                 fontsize=11, fontweight='bold', pad=10, color='black')
    ax.set_aspect('equal'); ax.axis('off')
    ax.set_xlim(-0.58, 0.58); ax.set_ylim(-0.55, 0.58)
    return bands_present


# ── precompute top-set per pannello + scala condivisa (size comparabili) ──
top_B = top_electrodes_plv(results_B)
top_C = top_electrodes_plv(results_C)
mag_max = max([m for _, m, _ in top_B.values()]
              + [m for _, m, _ in top_C.values()] + [1e-9])

# ── 2 panel: B (C0 ok/no) e C (C1 ok/no) ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 8))
fig.patch.set_facecolor('white')
all_bands = set()

all_bands |= plot_topo_panel(
    axes[0], top_B, mag_max=mag_max,
    title='B — C0 corretti vs sbagliati\n↑ corretti>sbagliati  ↓ sbagliati>corretti')
all_bands |= plot_topo_panel(
    axes[1], top_C, mag_max=mag_max,
    title='C — C1 corretti vs sbagliati\n↑ corretti>sbagliati  ↓ sbagliati>corretti')

patches = [mpatches.Patch(color=BAND_COLOR[b], label=f'{b} dominante')
           for b in ['alpha', 'beta', 'gamma'] if b in all_bands]
if patches:
    fig.legend(handles=patches, loc='lower center', ncol=len(patches),
               fontsize=11, facecolor='white', edgecolor='#CCCCCC',
               framealpha=1.0, bbox_to_anchor=(0.5, -0.02))

plt.suptitle(
    'EEG_23 — Hub PLV per-elettrodo (selezione per effect size)\n'
    'top-15 per |d| medio sulle coppie · colore = banda dominante · freccia = direzione netta',
    fontsize=13, fontweight='bold', y=1.02, color='black')
plt.tight_layout(rect=[0, 0.04, 1, 0.98])

save_path = FIG_DIR / 'eeg23_topo_hub_2panel.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
plt.close()

# riepilogo testuale
for name, top in [('C0 (B)', top_B), ('C1 (C)', top_C)]:
    print(f'\n=== {name} — top-{TOP_K} hub per |d| medio ===')
    for ch, (dom, mag, signed) in top.items():
        print(f'  {CHAN_NAMES_MNE[ch]:<5} {dom:<6} |d|={mag:.3f}  netto={signed:+.3f}')
print(f'\nSalvato in {save_path}')
